In [1]:
from dotenv import load_dotenv

load_dotenv()

True

## Summarize messages

In [2]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware

agent = create_agent(
    model="gpt-5-nano",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="gpt-4o-mini",
            trigger=("tokens", 100),
            keep=("messages", 1)
        )
    ],
)

In [3]:
from langchain.messages import HumanMessage, AIMessage
from pprint import pprint

response = agent.invoke(
    {"messages": [
        HumanMessage(content="What is the capital of the moon?"),
        AIMessage(content="The capital of the moon is Lunapolis."),
        HumanMessage(content="What is the weather in Lunapolis?"),
        AIMessage(content="Skies are clear, with a high of 120C and a low of -100C."),
        HumanMessage(content="How many cheese miners live in Lunapolis?"),
        AIMessage(content="There are 100,000 cheese miners living in Lunapolis."),
        HumanMessage(content="Do you think the cheese miners' union will strike?"),
        AIMessage(content="Yes, because they are unhappy with the new president."),
        HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?"),
        ]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'messages': [HumanMessage(content="Here is a summary of the conversation to date:\n\n## SESSION INTENT\nThe user is inquiring about various aspects of Lunapolis, the fictional capital of the moon, including its weather, population of cheese miners, and labor conditions related to the miners' union.\n\n## SUMMARY\nThe capital of the moon is Lunapolis. The current weather in Lunapolis features clear skies, with a high temperature of 120°C and a low of -100°C. The population of cheese miners in Lunapolis is 100,000. It is anticipated that the cheese miners' union may strike due to dissatisfaction with the new president.\n\n## ARTIFACTS\nNone\n\n## NEXT STEPS\nNone", additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='736ed205-001b-4a76-b2c1-b1086311db38'),
              HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?", additional_kwargs={}, response_metadata={}, id='e08cc766-1202-4055-a322-e759c37fa598'

In [4]:
print(response["messages"][0].content)

Here is a summary of the conversation to date:

## SESSION INTENT
The user is inquiring about various aspects of Lunapolis, the fictional capital of the moon, including its weather, population of cheese miners, and labor conditions related to the miners' union.

## SUMMARY
The capital of the moon is Lunapolis. The current weather in Lunapolis features clear skies, with a high temperature of 120°C and a low of -100°C. The population of cheese miners in Lunapolis is 100,000. It is anticipated that the cheese miners' union may strike due to dissatisfaction with the new president.

## ARTIFACTS
None

## NEXT STEPS
None


## Trim/delete messages

In [5]:
from typing import Any
from langchain.agents import AgentState
from langchain.messages import RemoveMessage
from langgraph.runtime import Runtime
from langchain.agents.middleware import before_agent
from langchain.messages import ToolMessage

@before_agent
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Remove all the tool messages from the state"""
    messages = state["messages"]

    tool_messages = [m for m in messages if isinstance(m, ToolMessage)]
    
    return {"messages": [RemoveMessage(id=m.id) for m in tool_messages]}

In [6]:
agent = create_agent(
    model="gpt-5-nano",
    checkpointer=InMemorySaver(),
    middleware=[trim_messages],
)

In [7]:
response = agent.invoke(
    {"messages": [
        HumanMessage(content="My device won't turn on. What should I do?"),
        ToolMessage(content="blorp-x7 initiating diagnostic ping…", tool_call_id="1"),
        AIMessage(content="Is the device plugged in and turned on?"),
        HumanMessage(content="Yes, it's plugged in and turned on."),
        ToolMessage(content="temp=42C voltage=2.9v … greeble complete.", tool_call_id="2"),
        AIMessage(content="Is the device showing any lights or indicators?"),
        HumanMessage(content="What's the temperature of the device?")
        ]},
    {"configurable": {"thread_id": "2"}}
)

pprint(response)

{'messages': [HumanMessage(content="My device won't turn on. What should I do?", additional_kwargs={}, response_metadata={}, id='7ae284cc-d94d-4b28-8ef4-13f6e48a8e43'),
              AIMessage(content='Is the device plugged in and turned on?', additional_kwargs={}, response_metadata={}, id='d52b377d-f914-4bca-bfc5-b9ccae03f2b4', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="Yes, it's plugged in and turned on.", additional_kwargs={}, response_metadata={}, id='434ee3c6-8c3c-4ae9-b9db-85912d6ea1b6'),
              AIMessage(content='Is the device showing any lights or indicators?', additional_kwargs={}, response_metadata={}, id='88ea4822-5429-4199-9ab9-3b395a54a775', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="What's the temperature of the device?", additional_kwargs={}, response_metadata={}, id='6d55b0bc-7ab2-4cf1-8041-fc4b6994efae'),
              AIMessage(content='I can’t read the device’s temperature from here. If the device

In [8]:
print(response["messages"][-1].content)

I can’t read the device’s temperature from here. If the device is off, there’s no temperature to report; if it’s on, you’ll need to read it with the right tools.

What kind of device is it (brand/model and whether it’s a phone, laptop/desktop, tablet, etc.)? I can give exact steps. In the meantime:

- If the device is hot to the touch or you’re concerned, power it off and unplug it, then let it cool in a ventilated area.
- Check for dust blocking vents and ensure fans are spinning when it’s powered.
- For measuring temps when it’s on:
  - Windows PC/laptop: use software like HWInfo or Core Temp to read CPU/GPU temps. Safe ranges vary, but idle ~30–40°C; under load up to ~70–90°C.
  - Mac: use a utility like Macs Fan Control or iStat Menus to view temps and control fans.
  - Android: many devices show battery temperature in Settings > Battery, or use an app like CPU-Z/AIDA64.
  - iPhone: temperature isn’t usually shown directly; keep it well below 35°C ambient; if it’s overheating, rest